# 🎯 Eval v14 STRONGEST LoRA — Full + Held-out comparison

**Mục đích:** Load v14 LoRA đã train (qwen3_cot_lora_v14_v4) và eval trên:
1. **TOÀN BỘ raw_data 318 records / 630 câu** (bao gồm training data)
2. **Held-out TEST split** (records chưa thấy)
3. **In mẫu 5 câu** từ train + 5 câu từ test để so gold vs prediction

→ So sánh với notebook_v14match (config nhẹ hơn r=16) khi cả hai chạy xong.


In [ ]:
%%capture
!pip install -q unsloth==2025.10.5 peft transformers

In [ ]:
import os, json, random, copy
from typing import List, Dict
from collections import Counter, defaultdict
from datasets import Dataset, DatasetDict

import torch
from unsloth import FastLanguageModel

SEED = 42
random.seed(SEED)
print("✅ Imports OK")

In [ ]:
def resolve_kaggle_path(*candidates):
    for p in candidates:
        if p and os.path.exists(p):
            print(f"  resolved: {p}")
            return p
    return candidates[0]

DATA_PATH = resolve_kaggle_path(
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
)
EXTERNAL_TEST_PATH = resolve_kaggle_path(
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
)
BASE_MODEL = resolve_kaggle_path(
    "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1",
    "/kaggle/input/qwen-3/transformers/8b/1",
)
LORA_PATH = resolve_kaggle_path(
    "/kaggle/input/notebooks/nguyenminhtric/v14-fine-tune/qwen3_cot_lora_v14_v4",
    "/kaggle/working/qwen3_cot_lora_v14_v4",
)

MAX_SEQ_LEN  = 2048
TEST_RATIO   = 0.10   # match v14match — same record-level split
VAL_RATIO    = 0.15
print("✅ Paths resolved")

In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"📦 Loaded {len(raw_data)} records / {sum(len(r['questions']) for r in raw_data)} questions")

In [ ]:
# Load Qwen + apply v14 LoRA (clean PEFT adapter)
print(f"Loading base Qwen + LoRA adapter at {LORA_PATH}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = LORA_PATH,   # Unsloth auto-detects adapter và load base
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("✅ Model loaded with v14 LoRA active")

In [ ]:
# System prompt — KHỚP với v14 training để inference accuracy tối đa
SYSTEM_PROMPT = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. \"From premise 3...\"). "
    "Then state your conclusion on a final line in the exact format: \"Final Answer: X\" "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)

def build_user_message(fol_list, nl_list, question):
    parts = ["### Premises"]
    for i, nl in enumerate(nl_list, 1):
        parts.append(f"P{i}. {nl}")
    parts.append("\n### Question")
    parts.append(question)
    return "\n".join(parts)

def extract_final_answer(text):
    import re
    if not text: return "UNPARSEABLE"
    m = re.search(r"Final\s*Answer\s*[:\-]?\s*([A-D]|Yes|No|Unknown)", text, re.I)
    if m:
        a = m.group(1).strip()
        return a.capitalize() if a.lower() in ("yes","no","unknown") else a.upper()
    # Fallback: look for last line
    for line in reversed(text.strip().splitlines()):
        m = re.search(r"\b(Yes|No|Unknown|[A-D])\b", line, re.I)
        if m:
            a = m.group(1).strip()
            return a.capitalize() if a.lower() in ("yes","no","unknown") else a.upper()
    return "UNPARSEABLE"

print("✅ Helpers ready")

In [ ]:
def predict(fol_list, nl_list, question, max_new_tokens=256, temperature=0.0):
    messages = [
        {"role":"system", "content": SYSTEM_PROMPT},
        {"role":"user",   "content": build_user_message(fol_list, nl_list, question)},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=(temperature > 0),
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("✅ predict() ready")

In [ ]:
# Record-level split — same seed as v14match for fair comparison
def split_records(raw_data, test_ratio=TEST_RATIO, val_ratio=VAL_RATIO, seed=SEED):
    rng = random.Random(seed)
    indices = list(range(len(raw_data)))
    rng.shuffle(indices)
    n = len(indices)
    n_test = int(n * test_ratio); n_val = int(n * val_ratio)
    return ([raw_data[i] for i in indices[n_test+n_val:]],
            [raw_data[i] for i in indices[n_test:n_test+n_val]],
            [raw_data[i] for i in indices[:n_test]])

train_records, val_records, test_records = split_records(raw_data)
train_entry_ids = {id(r) for r in train_records}
val_entry_ids   = {id(r) for r in val_records}
test_entry_ids  = {id(r) for r in test_records}

print(f"📊 Split (records | questions):")
print(f"   train: {len(train_records):3d} | {sum(len(r['questions']) for r in train_records):4d}")
print(f"   val  : {len(val_records):3d} | {sum(len(r['questions']) for r in val_records):4d}")
print(f"   test : {len(test_records):3d} | {sum(len(r['questions']) for r in test_records):4d}  ← held-out")

In [ ]:
# ============================================================
#  EVAL: FULL raw_data + per-split breakdown + sample print
# ============================================================
import random as _rng

print("📊 Eval v14 STRONGEST LoRA on full 318 records...")
print("=" * 65)

all_results = []
for entry_i, entry in enumerate(raw_data):
    eid = id(entry)
    split = "test" if eid in test_entry_ids else ("val" if eid in val_entry_ids else "train")
    for q_i, (q, gold_ans, gold_exp) in enumerate(
        zip(entry["questions"], entry["answers"], entry["explanation"])
    ):
        output = predict(entry["premises-FOL"], entry["premises-NL"], q,
                         max_new_tokens=256, temperature=0.0)
        pred = extract_final_answer(output)
        is_correct = (pred.strip().lower() == gold_ans.strip().lower())
        all_results.append({
            "entry_i": entry_i, "q_i": q_i, "split": split,
            "question": q, "gold": gold_ans, "pred": pred,
            "correct": is_correct, "gold_exp": gold_exp[:300],
        })
    if (entry_i+1) % 50 == 0:
        done = [r for r in all_results if r["entry_i"] <= entry_i]
        print(f"  [{entry_i+1:3d}/318] running: {100*sum(r['correct'] for r in done)/len(done):.1f}%")

train_results = [r for r in all_results if r["split"]=="train"]
val_results   = [r for r in all_results if r["split"]=="val"]
test_results  = [r for r in all_results if r["split"]=="test"]

def _acc(lst): return 100*sum(r["correct"] for r in lst)/len(lst) if lst else 0
full_acc, train_acc, val_acc, test_acc = _acc(all_results), _acc(train_results), _acc(val_results), _acc(test_results)

print("\n" + "=" * 65)
print("  v14 STRONGEST — ACCURACY COMPARISON")
print("=" * 65)
print(f"  FULL data (train+val+test): {sum(r['correct'] for r in all_results):4d}/{len(all_results):4d} = {full_acc:.1f}%")
print(f"  TRAIN only                : {sum(r['correct'] for r in train_results):4d}/{len(train_results):4d} = {train_acc:.1f}%")
print(f"  VAL only                  : {sum(r['correct'] for r in val_results):4d}/{len(val_results):4d} = {val_acc:.1f}%")
print(f"  TEST only (NEVER seen)    : {sum(r['correct'] for r in test_results):4d}/{len(test_results):4d} = {test_acc:.1f}%  ← SỐ THẬT")
print(f"\n  GAP (train - test): {train_acc - test_acc:+.1f}pp")
if train_acc - test_acc > 15:
    print(f"  ← OVERFIT NẶNG")
elif train_acc - test_acc > 7:
    print(f"  ← Overfit vừa")
elif train_acc - test_acc > 3:
    print(f"  ← Overfit nhẹ (normal cho dataset nhỏ)")
else:
    print(f"  ← OK generalize tốt")

# Per-class
print("\n  Per-class (TRAIN vs TEST):")
print(f"  {'Label':>10} {'Train':>8} {'Test':>8} {'N_test':>8}")
print(f"  {'-'*36}")
for label in sorted(set(r["gold"] for r in all_results)):
    tr = [r for r in train_results if r["gold"]==label]
    te = [r for r in test_results  if r["gold"]==label]
    ta = f"{_acc(tr):.0f}%" if tr else "n/a"
    ea = f"{_acc(te):.0f}%" if te else "n/a"
    print(f"  {label:>10} {ta:>8} {ea:>8} {len(te):>8}")

# Print samples
_seed = _rng.Random(42)
def print_examples(title, lst, n=5):
    print(f"\n{'='*65}\n  {title}\n{'='*65}")
    chosen = _seed.sample(lst, min(n, len(lst)))
    for i, r in enumerate(chosen, 1):
        tick = "✓" if r["correct"] else "✗"
        print(f"\n  [{i}] {tick} entry={r['entry_i']} q={r['q_i']}  Gold: {r['gold']:8s}  Pred: {r['pred']}")
        print(f"      Q: {r['question'][:100]}")
        print(f"      Gold exp: {r['gold_exp'][:150]}...")
        if not r["correct"]:
            print(f"      ⚠ WRONG: expected '{r['gold']}', got '{r['pred']}'")

print_examples("5 câu TRAIN (đã học)", train_results)
print_examples("5 câu TEST (chưa thấy)", test_results)

# Save
json.dump({
    "config": {"lora_path": LORA_PATH, "data_path": DATA_PATH},
    "summary": {"full_acc": full_acc, "train_acc": train_acc,
                "val_acc": val_acc, "test_acc": test_acc,
                "gap_train_test": train_acc - test_acc},
    "all_results": all_results,
}, open("/kaggle/working/eval_v14strongest_full.json","w",encoding="utf-8"),
   ensure_ascii=False, indent=2)
print(f"\n📁 Saved: /kaggle/working/eval_v14strongest_full.json")

In [ ]:
# Optional: external OOD test
RUN_EXTERNAL_TEST = True
if RUN_EXTERNAL_TEST and os.path.exists(EXTERNAL_TEST_PATH):
    print(f"\n📊 External OOD test: {EXTERNAL_TEST_PATH}")
    with open(EXTERNAL_TEST_PATH) as f: ext_data = json.load(f)
    ext_results = []
    for entry_i, entry in enumerate(ext_data):
        for q_i, (q, gold_ans, _) in enumerate(zip(entry["questions"], entry["answers"], entry["explanation"])):
            pred = extract_final_answer(predict(entry["premises-FOL"], entry["premises-NL"], q, max_new_tokens=256, temperature=0.0))
            ext_results.append({"correct": pred.strip().lower()==gold_ans.strip().lower(),
                                "gold": gold_ans, "pred": pred})
        if (entry_i+1) % 50 == 0:
            print(f"  [{entry_i+1:3d}/{len(ext_data)}] running: {100*sum(r['correct'] for r in ext_results)/len(ext_results):.1f}%")
    ext_acc = 100*sum(r["correct"] for r in ext_results)/len(ext_results)
    print(f"\n✅ EXTERNAL OOD: {sum(r['correct'] for r in ext_results)}/{len(ext_results)} = {ext_acc:.1f}%")
    if 'test_acc' in dir():
        print(f"\n  Test (in-dist)   : {test_acc:.1f}%")
        print(f"  External (OOD)   : {ext_acc:.1f}%")
        print(f"  OOD gap          : {test_acc - ext_acc:+.1f}pp  ← > 10pp = overfit pattern")
    json.dump({"acc": ext_acc, "n": len(ext_results), "results": ext_results},
              open("/kaggle/working/eval_v14strongest_external.json","w",encoding="utf-8"),
              ensure_ascii=False, indent=2)
